# ChipWhisperer 응용 연구 노트북 — Husky 와이어태핑을 통한 **전압 글리치(Voltage Glitching)** 동시 관측

## 능동(active) Lite + 수동(passive) Husky — 결합 위협 모델 실험 *(전압 글리치 고도화 판)*

---

### 🎯 노트북의 목표

본 노트북은 본 연구 그룹의 두 선행 자료를 결합·확장하고, **클럭 글리치 → 전압 글리치(crowbar VCC glitch)** 로 공격 기법을 고도화한 **응용편**입니다.

| 선행 자료 | 시나리오 | 본 노트북에서의 역할 |
|:----:|:----|:----|
| `FA_main.ipynb` | 단일 장치로 글리치 주입 | **글리치 파라미터 탐색·6단계 결과 분류 절차** 를 이어받음 |
| `Wiretapping4SCA.ipynb` | Lite(능동) + Husky(수동) 다중 장치 부채널 측정 | **다중 장치 동시 운용·트리거 분기 패턴** 을 이어받음 |
| `draft_Wiretapping4FA_claude.ipynb` *(원본)* | 결합 위협 모델 + **클럭** 글리치 | **결합 모델의 골격을 그대로 이어받되 글리치 종류를 전압으로 교체** |

### 🔬 본 고도화 판에서의 결합 위협 모델

```
┌───────────────────────────────────────────────────────────────────────────────┐
│  결합 시나리오 — Voltage Glitching 판                                              │
│                                                                               │
│   ChipWhisperer-Lite   ───  "능동 공격자" (active attacker)                       │
│      └─ 타겟 보드와 UART (SimpleSerial2) 로 통신                                    │
│      └─ 타겟 펌웨어를 컴파일·플래싱                                                  │
│      └─ 타겟에 깨끗한 시스템 클럭 공급 (HS2)  ← 클럭 글리치 아님                          │
│      └─ ★ NEW: 내장 HP/LP MOSFET 으로 **VDD 라인을 nanosecond 단위로 GND 단락**       │
│             (crowbar voltage glitch — Lite GLITCH SMA → CW308 GLITCH SMA 경로)    │
│                                                                               │
│   ChipWhisperer-Husky  ───  "은밀한 관측자" (covert observer)                      │
│      └─ 트리거 / 클럭 / **★VDD 라인★** 세 신호를 분기 측정 (wire-tap)                   │
│      └─ 통신·연산·글리치 제어에는 일체 개입하지 않음                                     │
│      └─ ★ KEY INSIGHT: 차동 모드로 **VDD 라인의 dip 형태(글리치 파형)** 를 직접 관측        │
│         (Lite 자신의 ADC 로는 자기 자신이 만든 glitch 의 실제 모양을 *알 수 없다*)          │
└───────────────────────────────────────────────────────────────────────────────┘
```

### 클럭 글리치 → 전압 글리치, 무엇이 달라지는가?

| 비교 항목 | 원본 (Clock Glitch) | **본 노트북 (Voltage Glitch)** |
|:----:|:----:|:----:|
| Lite 글리치 설정 함수 | `cglitch_setup()` | **`vglitch_setup()`** |
| Lite 글리치 출력 라인 | HS2 (디지털 클럭에 XOR 합성) | **GLITCH SMA (MOSFET 드레인, crowbar)** |
| 영향 받는 타겟 신호 | 시스템 클럭 (CLKIN) | **코어/IO 전원 (VDD)** |
| 공격 성립 조건 | 타겟이 외부 클럭에 동기 | 타겟의 디커플링 커패시터 제거/완화, VCC 노드 접근 |
| 발열·하드웨어 위험 | 거의 없음 | **MOSFET 발열·VRM 손상 위험** ← 안전 점검 필수 |
| 타겟 회복성 | 글리치 종료 시 즉시 복원 | **하드 리셋·brownout 빈발** ← 리셋 루틴 강화 |
| Husky 와이어태핑 대상 | CLKIN 라인 (글리치된 사각파 관찰) | **VDD 라인 (글리치 dip 직접 관찰)** ← 차동 측정 권장 |
| 적용 가능 타겟 범위 | 외부 클럭 동기 MCU 만 | **거의 모든 MCU/SBC** (Pi, ESP32, STM32 RDP 등) |
| 학술 적용 사례 | Riscure / NewAE 강좌 다수 | Raelize ESP32 Secure Boot, SEC Consult SECGlitcher, Trezor RDP 등 |

> 💡 **전압 글리치가 일반적으로 더 강력한 이유**
> 외부 클럭이 없는(혹은 내부 RC 오실레이터로 동작하는) 칩에도 적용 가능하고, 동일한 칩에 대해 더 다양한 instruction 에 fault 를 일으킬 수 있습니다.
> 다만 그 대가로 **하드웨어 개조(디커플링 커패시터 제거)** 와 **회복성 관리**가 클럭 글리치보다 훨씬 까다롭습니다 → 본 노트북은 그 차이를 코드·배선 양쪽에서 반영합니다.

### 노트북 구조 (고도화 판)

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **0단계** | **★ 하드웨어 배선·점퍼·커패시터 개조 가이드 (대폭 확장) ★** | 검증된 물리 셋업 |
| **1단계** | 라이브러리 임포트 + 다중 장치 동시 연결 | `lite_scope`, `husky_scope` |
| **2단계** | Lite ↔ 타겟 UART 채널 (SimpleSerial2) 확보 | `target` 객체 |
| **3단계** | Lite 경유 타겟 펌웨어 빌드·플래싱 | 플래싱 완료된 타겟 |
| **4단계** | Golden Model 통신 검증 | `expected_ret` |
| **5단계** | **Lite 측 전압 글리치 모듈 설정 (`vglitch_setup`) + FIA 용 1 sample/clock 정렬** | crowbar 활성화된 Lite |
| **6단계** | Husky 측 와이어태핑 환경 (외부 클럭 PLL + ★차동 VDD 측정★ + 트리거) | 측정 준비 완료된 Husky |
| **7단계** | 베이스라인 동시 캡처 + 글리치 시점 후보 식별 | 글리치 없는 정상 파형 |
| **8단계** | 글리치 파라미터 탐색 + **VDD dip 파형** 동시 측정 + ★MOSFET 보호 휴식★ | 결과 6단계 분류 + 코드별 VDD 파형 |
| **9단계** | 통계 분석으로 최적 글리치 파라미터 도출 | `(i_offset, i_width)` 최빈값 |
| **10단계** | Bokeh 시각화 — 파라미터 분포 + ★실측 VDD 글리치 dip 비교★ | 인터랙티브 그래프 |
| **마무리** | 안전한 다중 장치 자원 해제 (MOSFET 비활성화 우선) | 깨끗한 종료 |

> 본 자료는 원본 노트북이 이미 다룬 기본기 (SimpleSerial, `my_fsr_cmd` 헬퍼, 6단계 결과 분류, Bokeh 사용법 등) 의 반복 설명을 최소화하고, **전압 글리치 고유의 하드웨어 셋업·운용·관찰 포인트** 에 집중합니다.


---

# 🔧 0단계 — 하드웨어 배선·점퍼·커패시터 가이드 (★ 코드 실행 전 반드시 점검 ★)

> **이 단계의 목표**
> 전압 글리치 공격은 **물리적 배선·점퍼·커패시터 상태가 성패의 90% 를 좌우** 합니다.
> 본 단원은 셀 실행이 없는 **하드웨어 점검 단원** 입니다. 모든 코드 셀을 실행하기 *전에* 본 단원의 체크리스트를 모두 통과해야 합니다.

---

## 0.1 필요한 케이블 / 어댑터 인벤토리

| 번호 | 케이블 / 부품 | 규격 | 용도 |
|:----:|:----|:----|:----|
| ① | USB-C 케이블 | USB-C ↔ USB-A or C | PC ↔ **Husky** |
| ② | Micro-USB 케이블 *(Lite 리비전에 따라 USB-A/C 도 가능)* | — | PC ↔ **Lite** |
| ③ | 20-pin IDC 리본 케이블 (스트레이트, 1:1 핀맵) | 2.54mm 피치, 길이 ~10cm | **Lite 20-pin** ↔ **CW308 J3 (Main 20-pin)** |
| ④ | SMA male-male 동축 케이블 (RG-316 권장) | 50Ω, 길이 ~15cm | **★ Lite "GLITCH" SMA ↔ CW308 "GLITCH" SMA (Crowbar 경로)** |
| ⑤ | SMA male-male 동축 케이블 | 50Ω, 길이 ~15cm | CW308 "MEAS"(SHUNTL) ↔ Husky **"Measure POS"** SMA *(선택: 션트 전류 동시 관찰)* |
| ⑥ | **MCX female ↔ SMA male** 어댑터 (또는 변환 케이블) | 50Ω | CW308 "CLKIN" 노드 분기 → **Husky "AUX" MCX** *(클럭 PLL 동기용)* |
| ⑦ | **트리거 Y-분기 (SMA 또는 듀폰)** | — | CW308 TRIG (GPIO4) → **Lite IO4 + Husky 전면 20-pin D0** (1:2 분기) |
| ⑧ | **★ VDD 직결 프로빙 와이어 (단심 AWG28~30, ~5cm)** | — | 타겟 VDD 핀(또는 디커플링 캡 양단) → **Husky "Measure POS"** SMA 의 신호 측 |
| ⑨ | GND 동선 (브레이드 또는 단심) | 짧고 굵게 | 타겟 GND → Husky SMA 의 GND shell (낮은 인덕턴스가 핵심) |
| ⑩ | SMD 핀셋 + 0402/0603 제거용 인두 (~280°C) | — | **디커플링 커패시터 제거** 작업 |

> ⚠️ **④⑤⑥⑧이 본 노트북에서 새로 추가/변경되는 라인입니다.**
> 원본(클럭 글리치) 노트북은 ④(GLITCH SMA) 라인이 *없었고*, ⑥(AUX MCX) 라인이 *글리치된 클럭* 을 봤습니다.
> 본 노트북에서는 ④가 **공격 주입 라인**, ⑥은 **순수 클럭(글리치 없음)** , ⑧이 **공격 관찰 라인** 으로 역할이 명확히 분리됩니다.


## 0.2 전체 배선 다이어그램

```
                              ┌──────────────────┐
                              │   Host PC        │
                              │   (Jupyter)      │
                              └────┬─────────┬───┘
                                ① │       ② │  USB
                          ┌────────┘         └────────┐
                          ▼                            ▼
                  ┌───────────────────┐        ┌──────────────────────┐
                  │  ChipWhisperer    │        │  ChipWhisperer       │
                  │  Husky (수동 관측)  │        │  Lite   (능동 공격)   │
                  └────┬──────┬───────┘        └────┬──────┬──────┬───┘
                       │      │                     │      │      │
       MEAS POS SMA ◄──┘      │              GLITCH ┘      │      │
       (또는 Differential)    │               SMA ④        │      │
                              │                            │      │
       AUX MCX (입력) ◄───┐   │  D0 (전면 20-pin)          │      │
       ⑥                  │   │  ⑦ (트리거 입력)          │      │
                          │   │                            │      │
                          │   └────────┐                   │      │
                          │            ▼                   ▼      ▼
                          │       ┌──────────┐         ┌──────────┐
                          │       │ Trig     │         │ Lite     │
                          │       │ Y-분기   │         │ 20-pin   │
                          │       │ ⑦       │         │ ③(케이블)│
                          │       └────┬─────┘         └─────┬────┘
                          │            │                     │
                          │            ▼ TRIG (GPIO4)        ▼ J3
                          │       ┌─────────────────────────────────┐
                          │       │                                 │
                          │       │      CW308 UFO Main 보드        │
                          │       │      (점퍼 설정은 0.4 참조)      │
                          │       │                                 │
                          │       │   ┌──── GLITCH SMA  ◄────────── ④
                          │       │   │     (input from Lite)        │
                          │       │   │                              │
                          │       │   │     CLKIN 노드 ◄─ HS2 (Lite) │
                          │       │   │         │                    │
                          │       └───┘         └─ MCX 분기 ⑥ ──────┘
                          │                            ▲(클럭 PLL 동기)
                          │                            │
                          │       MEAS SMA (SHUNTL) ──⑤────► Husky MEAS POS
                          │                            
                          │                                 
                          │       ┌──────────────────────────┐
                          │       │ CW308T_STM32F3 타겟 보드  │
                          │       │ (UFO 메인 위에 장착)       │
                          │       │                          │
                          │       │ STM32F303 칩              │
                          │       │   VDD ◄──┐               │
                          │       │   GND ─┐ │               │
                          │       └────────┼─┼──────────────┘
                          │                │ │
                          └─── 직결 프로빙 ⑧⑨ (와이어태핑)
                                          │ │
                                          └─┴─► Husky MEAS POS (or Differential)
                                          ※ VDD dip 직접 관찰용 — 본 노트북의 핵심
```

**핵심 신호 흐름 (전압 글리치 판)**

```
[글리치 주입 경로 — 능동]
   Lite 내부 HP/LP MOSFET ─► Lite GLITCH SMA ─④─► CW308 GLITCH SMA
                                                  └─► SJ 경유 ─► VDD 노드
                                                         │
                                                         ▼
                                                  [타겟 STM32F303 VDD]
                                                         │
                                                         ▼
                                                  GLITCH 시 nanosecond 단위 GND 단락 (crowbar)

[글리치 관찰 경로 — 수동]
   타겟 VDD 핀 ──직결 와이어⑧──► Husky MEAS POS SMA
                                  ▲
                                  └─ Husky 전면 ADC 차동 입력 (혹은 단일 종단)
                                          │
                                          ▼
                                  4× 오버샘플링으로 dip 형태 직접 관찰

[클럭 / 트리거 — 변경 없음, 단 클럭은 더 이상 "글리치되지 않음"]
   Lite HS2 ─► CW308 CLKIN ─⑥─► Husky AUX MCX  (PLL 동기 — 깨끗한 클럭이라 안정적)
   CW308 GPIO4 (TRIG) ─⑦분기─► Lite IO4  +  Husky D0
```


## 0.3 Lite 측 연결 상세 (능동 글리처)

### 0.3.1 Lite 20-pin 커넥터 핀맵 (③ 케이블)

| Pin | 명칭 | 방향 (Lite 기준) | 본 노트북 용도 |
|:----:|:----|:----:|:----|
| 1 | VREF | I/O | CW308 의 VREF (3.3V 기준) — *모니터링용* |
| 2 | nRST | OUT | **타겟 리셋** (`scope.io.nrst`) — 글리치 후 회복에 필수 |
| 3 | PDID / SWDIO | I/O | 펌웨어 플래싱 (SWD) |
| 4 | VCC | OUT | **타겟에 3.3V 공급** (Lite → CW308) |
| 5 | PDIC / SWCLK | OUT | 펌웨어 플래싱 |
| 6 | GND | — | 공통 GND |
| 7 | (nc) | — | — |
| 8 | GND | — | 공통 GND |
| 9 | (nc) | — | — |
| 10 | GND | — | 공통 GND |
| 11 | **TIO1 / TX→** | OUT | SimpleSerial2 **TX (Lite → Target)** |
| 12 | **TIO2 / ←RX** | IN | SimpleSerial2 **RX (Target → Lite)** |
| 13 | TIO3 | I/O | (미사용) |
| 14 | **TIO4 / TRIG** | IN | **★ 트리거 입력 (GPIO4 → Lite IO4)** ⑦ |
| 15 | (nc) | — | — |
| 16 | **HS2 / CLKOUT** | OUT | **★ 시스템 클럭 출력 (CW308 CLKIN 으로)** ← 클럭 글리치 아님, 깨끗함 |
| 17 | (nc) | — | — |
| 18 | (nc) | — | — |
| 19 | GND | — | 공통 GND |
| 20 | GND | — | 공통 GND |

> 💡 **핵심: 20-pin 케이블만으로는 글리치가 전달되지 않습니다.**
> 클럭 글리치는 HS2(pin 16) 위에 XOR 로 합성되어 20-pin 케이블만으로 전달되지만, **전압 글리치는 별도의 SMA 케이블(④) 이 반드시 필요** 합니다. ★

### 0.3.2 Lite "GLITCH" SMA — 본 노트북에서 새로 활성화되는 라인

Lite 측면(또는 후면)에 위치한 **GLITCH** 라벨의 SMA 커넥터:

```
┌─────────────────── Lite 내부 회로 (개념) ───────────────────┐
│                                                            │
│   FPGA glitch_hp/lp 신호                                    │
│        │                                                   │
│        ├─► [HP MOSFET (큰 N-MOS)]   ┐                       │
│        │      └─ Drain ──────────┐  │                       │
│        │                          │  ├─► GLITCH SMA 중심핀  │
│        └─► [LP MOSFET (작은 N-MOS)]│  │   (Crowbar 출력)    │
│               └─ Drain ───────────┘  │                       │
│                                       │                       │
│                          GND ─────────┴─► GLITCH SMA shell  │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

| MOSFET | 특성 | 본 노트북 권장 |
|:----:|:----|:----|
| **HP (High Power)** | 큰 전류, 강한 풀-다운, **발열 많음** | STM32F303 같은 일반 MCU 에서 권장 (`scope.io.glitch_hp = True`) |
| **LP (Low Power)** | 작은 전류, 약한 풀-다운 | 저전력 타겟·미세 조정용 (둘 다 켜는 조합도 가능) |

> ⚠️ **MOSFET 보호 — 가장 중요한 안전 수칙**
> - `glitch.repeat` 값이 너무 커지면 MOSFET 이 *지속적으로* 단락되어 **수 초 내에 소손** 됩니다.
> - 전체 width × repeat 가 클럭 1주기를 넘지 않도록 8단계에서 보호 로직을 둡니다.
> - 5분 이상 연속 탐색 시 **MOSFET 표면 온도가 60℃ 이상이면 즉시 휴식** (`time.sleep(30)`)


## 0.4 CW308 UFO 메인 보드 점퍼 설정 ★★★

> **본 노트북에서 가장 실수가 잦은 영역입니다.** 점퍼 한 개를 잘못 두면 글리치가 *전혀* 전달되지 않거나, *항상* 단락 상태가 되어 보드가 망가질 수 있습니다.

### 0.4.1 전원 공급 점퍼

| 점퍼 | 위치 | 본 노트북 설정 | 의미 |
|:----:|:----|:----:|:----|
| **J1** | 전원 선택기 | **20-pin connector** (Lite 의 VCC 핀 → CW308) | 외부 전원 어댑터 사용 시 위험 — Lite 공급 권장 |
| **SJ1** | VREF 선택 | **3.3V** | VREF = 3.3V (STM32F3 의 디지털 I/O) |
| **SJ2** | (보드 리비전에 따라 — 일부 보드에만 존재) | 데이터시트 기본값 | 일반적으로 변경 불필요 |

### 0.4.2 ★ 글리치 경로 점퍼 — 본 노트북의 핵심 ★

CW308 UFO 보드는 **GLITCH SMA 입력으로 들어온 신호** 를 **타겟의 VDD 또는 VDDINT 노드** 로 라우팅하기 위한 점퍼들을 제공합니다.

| 점퍼 | 본 노트북 설정 | 의미·주의사항 |
|:----:|:----:|:----|
| **SJ4** *(또는 보드 리비전에 따라 다른 번호)* | **OPEN (제거)** | VDD 공급 라인 상의 **LC 저역필터를 제거** — 글리치가 평탄화되지 않게 함. 이걸 OPEN 안 하면 글리치가 거의 무력화됩니다. |
| **SJ5** | **CLOSED (점퍼)** | GLITCH SMA → 타겟 VDD 노드 경로 활성화. **이게 OPEN 이면 글리치가 전혀 안 들어갑니다.** |
| **SJ6 / SJ7** *(VDD 소스 선택)* | **SHUNT 경유** | 션트 저항(R12 등) 을 거쳐 VDD 공급 — Husky 가 션트 양단을 측정해 전력 분석 가능 |
| (그 외 분리형 power island 점퍼) | 타겟 데이터시트 권장값 | VDDA, VDDIO 등 분리 공급이 필요한 칩만 해당 |

> ⚠️ **SJ 번호는 CW308 보드 리비전마다 다를 수 있습니다.**
> 본 노트북은 일반적인 매핑을 기준으로 작성되었으나, 실제 작업 전에는 다음을 반드시 확인하세요:
> 1. **본인 보드의 실크스크린에서 "GLITCH" 패드 → 타겟 VDD 까지의 경로**를 멀티미터 도통 모드로 추적
> 2. **NewAE 의 CW308 메인보드 스키매틱 PDF** (rtfm.newae.com 에서 본인 보드 리비전에 맞는 페이지)
> 3. 점퍼 변경 전후로 **VDD 노드의 정전 상태** 가 어떻게 바뀌는지 확인 (글리치 OFF 시 안정된 3.3V 가 유지되어야 함)

### 0.4.3 시리얼/트리거 경로 점퍼

| 점퍼 | 본 노트북 설정 | 의미 |
|:----:|:----:|:----|
| **JP4 / JP5** *(또는 보드별 명칭)* | 표준 STM32 UART 핀 매핑 | TX/RX 가 STM32F3 의 PA9/PA10 에 도달 |
| **TRIG (J7?)** | 그대로 사용 | GPIO4 (PB4 등) 가 TRIG 라벨 SMA 또는 헤더로 노출 |

### 0.4.4 점퍼 설정 체크리스트 (셀 실행 전 ★)

본 노트북의 모든 코드 셀을 실행하기 전, 아래 항목을 **반드시** 모두 확인하세요:

- [ ] J1: 20-pin connector 전원 (Lite 가 VCC 공급)
- [ ] SJ1: VREF = 3.3V
- [ ] **SJ4 (LC 필터): OPEN ★ (점퍼 제거)**
- [ ] **SJ5 (GLITCH 라우팅): CLOSED ★ (점퍼 삽입)**
- [ ] SJ6/SJ7: SHUNT 경유 설정
- [ ] UART 점퍼: STM32 PA9/PA10 에 도달하는 경로
- [ ] **모든 외부 전원(어댑터) 제거** — Lite 만 전원 공급
- [ ] 멀티미터로 **VDD 노드 ↔ Lite GLITCH SMA 중심핀** 의 *off 상태 저항* 확인 (MOSFET 이 OFF 일 때 ≥ 1MΩ 이어야 함, 0Ω 면 단락으로 보드 손상)


## 0.5 디커플링 커패시터 처리 ★

전압 글리치가 *효과적으로* 동작하려면, **VDD 노드의 커패시턴스가 충분히 작아야** 합니다.
디커플링 커패시터(C12, C13 등의 100nF + 10μF 조합)가 그대로 있으면, MOSFET 이 잠깐 단락해도 커패시터가 즉시 전압을 복원해 글리치 효과가 사라집니다.

### 0.5.1 제거 권장 목록 (CW308T_STM32F3 기준)

| 부품 (전형적 위치) | 용량 | 처리 |
|:----:|:----|:----|
| C1 (10μF 벌크) | 대용량 | **★ 반드시 제거** — 가장 큰 글리치 평탄화 요인 |
| C2~C5 (100nF 디커플) | 소용량 (4개 안팎) | 권장: **모두 제거** (혹은 최소 2개만 남겨 안정성과 글리치 효과 균형) |
| VDDA 디커플 (분석 시 무관) | — | 보존 가능 (analog 도메인은 글리치 대상 아님) |

> ⚠️ **부품 번호는 본인 보드 실측 기준** 으로 식별하세요. 일반적인 STM32 미니멈 회로 가이드(ST AN2586) 의 디커플 패턴을 따른 보드가 많습니다.

### 0.5.2 제거 절차

1. 보드 전원 완전 차단 (USB·어댑터 모두 분리)
2. ESD 손목 스트랩 착용
3. **인두 온도 320~350°C**, 0402/0603 칩 양 끝 패드를 동시에 가열 (트윈 팁이 이상적, 없으면 빠르게 좌우 번갈아)
4. SMD 핀셋으로 들어 올리기
5. 패드 잔여 솔더를 솔더윅으로 정리
6. **멀티미터로 VDD ↔ GND 단락 여부 확인** (절대 단락이면 안 됨)
7. 제거한 캡들은 **라벨 봉투에 보관** — 실험 후 복원하려면 다시 납땜 필요

### 0.5.3 비파괴 대안 — 외부 캡 추가만 살짝 하기

본격 제거가 부담스러울 경우:
- 디커플 커패시터는 그대로 두고, **GLITCH SMA 와 가까운 위치에 직렬로 작은 인덕터(10~100nH)** 또는 **0Ω 저항을 ferrite bead 로 교체** 해 글리치 펄스를 통과시키되 캡으로의 전류 경로는 다소 차단
- 효과는 캡 제거보다 약하나, 보드 복원이 쉽습니다 → 학습·실험용으로만 권장


## 0.6 Husky 측 와이어태핑 배선 상세 (수동 관측자)

### 0.6.1 Husky 전면 / 측면 커넥터 개요

```
┌────────────── Husky 전면 ──────────────┐    ┌── 측면 ──┐
│                                        │    │          │
│  [USERIO 20-pin 헤더]                   │    │ MEAS POS │── ⑤ or ⑧
│   ├ D0 (트리거 입력 ⑦)                 │    │ MEAS NEG │── (차동 시 GND 또는 VREF)
│   ├ D1 ~ D7 (디지털 로직 분석기)         │    │ GLITCH   │── (본 노트북 미사용)
│   └ ...                                │    │ TRIG OUT │── (외부 트리거 송출 시)
│                                        │    │ AUX MCX  │── ⑥ (외부 클럭 입력)
│  [JTAG/SWD 헤더]  (미사용)             │    │          │
│                                        │    │          │
└────────────────────────────────────────┘    └──────────┘
```

### 0.6.2 Husky 측 핵심 4개 연결

| 라인 | 출처 (타겟/CW308) | 입력 (Husky) | 본 노트북 역할 |
|:----:|:----|:----|:----|
| **클럭 동기** | CW308 CLKIN 노드 | **AUX MCX** ⑥ | 외부 PLL 소스 (`extclk_aux_io`) — 깨끗한 클럭이라 안정적 |
| **트리거** | CW308 GPIO4 (TRIG) ⑦ | **전면 USERIO D0** | Lite IO4 와 Y-분기 — 두 스코프 동시 정렬 |
| **VDD 직접 측정 ★** | 타겟 VDD 핀 ⑧ | **측면 MEAS POS** (Pos 핀, 차동 시 NEG 도 사용) | **글리치 dip 형태 직접 관찰** ← 본 노트북의 핵심 와이어태핑 |
| **(선택) 션트 전류** | CW308 MEAS SMA (SHUNTL) ⑤ | (대안: MEAS POS) | 션트 양단 측정 — 전류 변화 관찰 (한 번에 둘 중 하나만 가능) |

> 🔬 **VDD 직접 측정 vs 션트 측정 — 무엇을 골라야 하나?**
> - **글리치 *파형* 자체** 를 보고 싶다 → **VDD 직접 측정 (⑧)** 권장. dip 의 깊이·폭·rise time 이 그대로 보임
> - **글리치 효과로 인한 *전류 변화*** 를 보고 싶다 → 션트 측정 (⑤). 글리치 자체 모양은 잘 안 보임
> - 본 노트북은 ⑧을 기본으로 사용하고, 션트는 *옵션* 으로 남겨 둡니다.

### 0.6.3 ⑧ VDD 직접 측정 — 솔더링 가이드

```
타겟 STM32F303
   VDD pin (예: pin 1 — VBAT/VDD)
        │
        ├─── AWG 28~30 단심선 (길이 5cm 이하 권장)
        │      └─ 반대쪽에 SMA male 의 중심핀 솔더
        │
   GND pin (인접 GND)
        │
        ├─── 굵은 GND 동선 (브레이드 권장)
        │      └─ SMA male shell 에 솔더
        │
        └─► Husky MEAS POS SMA 로 직결
```

| 항목 | 권장값 | 이유 |
|:----|:----:|:----|
| 와이어 길이 | **≤ 5cm** | 인덕턴스 ↓ → 글리치 고주파 성분 보존 |
| 와이어 굵기 | AWG 28~30 | 너무 굵으면 패드 손상, 너무 가늘면 단선 |
| GND 길이 | 신호선과 동일/짧게 | 차동 인덕턴스 최소화 |
| 솔더링 위치 | 디커플 캡 자리 (제거 후) 또는 칩 핀에 직접 | 캡 패드는 굵고 안정적, 칩 핀은 가깝지만 손상 위험 |

### 0.6.4 (선택) Husky 차동 측정 — 더 깨끗한 글리치 파형

Husky 의 MEAS 입력은 차동 입력을 지원합니다.
- **POS** = VDD 직결
- **NEG** = 옆의 안정된 3.3V 기준 (혹은 GND)

차동 측정 시 공통 모드 잡음이 제거되어 글리치 dip 이 훨씬 선명하게 보입니다. 본 노트북은 단일 종단(POS only) 을 기본으로 두지만, 6단계 코드에서 `husky_scope.adc.input = 'analog_diff'` 옵션 주석을 풀어 차동으로 전환할 수 있습니다.


## 0.7 트리거 Y-분기 ⑦ 제작 가이드

CW308 의 GPIO4(TRIG) 신호 한 가닥을 **Lite IO4 + Husky D0** 두 곳에 동시에 입력해야 두 스코프가 *동일 트리거 엣지* 에 정렬 캡처할 수 있습니다.

### 0.7.1 권장 방식 — 듀폰 Y-분기

```
CW308 GPIO4 (TRIG) ─┬─► Lite TIO4 (20-pin pin 14)
                    │
                    └─► Husky USERIO D0 (전면 20-pin)
```

- 단순 듀폰 점퍼선 두 가닥을 한 끝에서 묶어 사용
- 신호 무결성에 민감하지 않은 단발성 트리거이므로 단순 분기로 충분

### 0.7.2 (정밀하게 하고 싶다면) SMA Y-스플리터 + 종단 50Ω

- 트리거가 고주파(>50MHz) 라면 SMA T-분기 + 한쪽 종단을 50Ω 로 매칭
- 본 노트북의 트리거는 한 번 토글되는 GPIO 이므로 위 단순 듀폰으로 충분


## 0.8 ★ 최종 안전 점검 체크리스트 (셀 실행 전 반드시 확인)

| 단계 | 점검 항목 | 통과 기준 |
|:----:|:----|:----|
| ① 전원 | 외부 어댑터·USB hub 의 추가 전원 모두 분리 | Lite USB 한 줄로만 전원 공급 |
| ② USB | Lite + Husky 가 PC 에 별도 USB 포트로 연결 (허브 X) | `cw.list_devices()` 가 둘 다 인식 |
| ③ 20-pin | Lite ↔ CW308 20-pin 케이블, **방향 표시(빨간 줄) 일치** | Pin 1 위치 확인 |
| ④ GLITCH SMA | Lite GLITCH ↔ CW308 GLITCH, SMA 너트 손으로 끝까지 (공구 사용 금지) | 흔들림 없음 |
| ⑤ MEAS SMA | Husky MEAS POS ↔ VDD 직결 와이어 (또는 션트), GND 공통화 | 도통 OK, 단락 X |
| ⑥ AUX MCX | Husky AUX MCX ↔ CW308 CLKIN 분기, 잠금 확인 | 흔들림 없음 |
| ⑦ Trigger | Y-분기로 Lite IO4 + Husky D0 동시 입력 | 멀티미터로 분기 도통 확인 |
| ⑧ 점퍼 (★) | SJ4=OPEN, SJ5=CLOSED, SJ6/7=SHUNT 경유 | 0.4.4 체크리스트 |
| ⑨ 디커플 (★) | 제거 대상 캡 모두 제거됨 + VDD-GND 단락 없음 | 멀티미터 |
| ⑩ MOSFET 상태 | **글리치 비활성 시** Lite GLITCH SMA ↔ GND 저항 = ∞ | 단락이면 즉시 정지 |
| ⑪ ESD | 손목 스트랩 착용, 보드 직접 접촉 최소화 | — |
| ⑫ 종료 절차 숙지 | 마무리 단원의 `disconnect_all_devices()` 가 MOSFET 비활성 → target.dis() → scope.dis() 순서임을 확인 | — |

> ⚠️ **위 12개 항목 중 하나라도 미충족이면 코드 셀을 실행하지 마세요.**
> 잘못된 셋업으로 인한 보드 파손은 거의 모든 사례에서 위 12개 중 하나의 누락에서 발생합니다.

---

이상 0단계의 하드웨어 점검이 완료되었다면, 다음 1단계부터의 코드 셀을 실행합니다.
실제 실험 중 글리치가 *전혀* 안 들어가거나 *항상* freezing 이 발생한다면, 코드보다 **본 0단계 체크리스트로 먼저 돌아오세요.**

---


# 📦 1단계 — 라이브러리 임포트 및 다중 장치 연결

> **이 단계의 목표**
> ChipWhisperer Python API 와 보조 헬퍼를 로드하고, **Lite + Husky 두 장치를 시리얼 넘버 기반** 으로 동시에 객체화합니다.
> 원본 노트북과 동일하지만, 본 노트북에서는 1단계 진입 *이전에* 0단계의 하드웨어 점검이 완료되어 있어야 합니다.

---

### 1.1 헬퍼 로드 및 상수 정의

`My_script.ipynb` 는 `my_fsr_cmd()`, Bokeh 임포트, 시각화 함수(`plot_t`), 시드 고정 등을 일괄 로드합니다.


In [ ]:
# 사전 정의된 헬퍼 (my_fsr_cmd, plot_t 등) 로드
%run My_script.ipynb

import chipwhisperer as cw
import logging
import time
import numpy as np
import pandas as pd
import scipy as sp
from tqdm.notebook import trange, tqdm

# ─────────────────────────────────────────
# 타겟 / 펌웨어 관련 상수
# ─────────────────────────────────────────
PLATFORM      = 'CW308_STM32F3'   # 타겟 보드 종류
SCOPETYPE     = 'OPENADC'         # 캡처 장치 (Lite/Husky 공통)
CRYPTO_TARGET = 'NONE'            # 자체 펌웨어 (XOR 루프) 사용
SS_VER        = 'SS_VER_2_1'      # SimpleSerial 프로토콜 버전


### 1.2 두 장치 동시 검출 및 객체 분리

`cw.list_devices()` 가 반환하는 시리얼 넘버를 명시해 `cw.scope(sn=...)` 로 연결합니다.

> 💡 이 셀은 원본 노트북과 **완전히 동일** 합니다 — 장치의 *역할 분기* 는 5/6단계에서 일어납니다.


In [ ]:
def connect_all_devices() -> dict:
    '''연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환'''
    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}

    for device in device_list:
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes

# PC에 연결된 모든 장치 인식 및 할당
scopes = connect_all_devices()
lite_scope  = scopes["ChipWhisperer_Lite"]
husky_scope = scopes["ChipWhisperer_Husky"]


---

# 🔌 2단계 — Lite 를 통한 타겟 보드 통신 채널 확보

> **이 단계의 목표**
> 타겟(STM32F303) 과의 **모든 시리얼 통신을 Lite 가 전담** 합니다.
> Husky 는 통신에 일체 개입하지 않으며, 본 노트북의 모든 단계에서 트리거·클럭·**VDD 라인** 만 외부에서 관측합니다.

---

> ⚠️ **`cw.target(husky_scope, ...)` 를 호출하지 말 것.**
> 채널이 Husky 로 바뀌어 본 시나리오의 의도와 어긋납니다.


In [ ]:
# Lite를 통한 타겟 보드 연결 설정
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("지원되지 않는 SimpleSerial 버전입니다.")

try:
    target = cw.target(lite_scope, target_type)
    print("\n[✓] ChipWhisperer_Lite 에 타겟 보드 연결 성공")
except Exception as e:
    print(f"\n[✗] ChipWhisperer_Lite 에 타겟 보드 연결 실패: {e}")


---

# 🛠 3단계 — 펌웨어 빌드 및 Lite 경유 타겟 프로그래밍

> **이 단계의 목표**
> `simpleserial_main/` 의 펌웨어를 STM32F303 용으로 컴파일하고, **Lite 의 SWD 인터페이스** 로 타겟에 플래싱합니다.

---

> 💡 **`lite_scope.default_setup()` 의 부수 효과**
> 이 호출은 Lite 의 게인·ADC·트리거 모드를 표준값으로, **HS2 를 타겟 클럭 공급원** 으로 설정합니다.
> 원본 노트북에서는 5단계의 `cglitch_setup()` 이 HS2 모드를 *"클럭 + 글리치 XOR"* 로 변경했지만,
> **본 노트북(전압 글리치)에서는 HS2 는 끝까지 깨끗한 클럭만 출력합니다** — 글리치는 GLITCH SMA 의 MOSFET 경로로 나갑니다.


In [ ]:
# 1. 펌웨어 컴파일
print("펌웨어 컴파일 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 완료")

# 2. 프로그래머 선택
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("프로그래머가 지원되지 않는 플랫폼입니다.")

# 3. 펌웨어 플래싱 (Lite 가 프로그래머 역할)
lite_scope.default_setup()
try:
    hex_path = f"simpleserial_main/simpleserial-base-{PLATFORM}.hex"
    cw.program_target(lite_scope, prog, hex_path)
    print(f"[✓] {PLATFORM} 타겟 보드에 프로그램 업로드 완료")
except Exception as e:
    print(f"[✗] 펌웨어 프로그램 실패: {e}")

# 4. 빌드 산출물 정리
print("펌웨어 컴파일 클린 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}", "clean"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 클린 완료")


---

# ✅ 4단계 — Golden Model 통신 검증

> **이 단계의 목표**
> 글리치를 *주입하지 않은* 상태에서 Lite ↔ 타겟 통신·연산이 정상인지 확인합니다.
> 본 검증을 통과한 이후 단계에서 보이는 비정상 결과는 모두 **글리치의 영향**으로 해석할 수 있습니다.

---

```c
for (i = 0; i < global_len; i++) {
    output[i] = key[i] ^ plaintext[i];
}
```

> ⚠️ **전압 글리치 시 추가로 점검할 것**
> 만약 4단계 검증이 일부 시행에서 random 하게 실패한다면, **MOSFET 의 OFF-state 누설 / 부정확 배선** 으로 VDD 가 미세하게 흔들리고 있다는 신호입니다 → 0단계로 복귀해 점퍼·디커플·MOSFET OFF 저항을 재확인.


In [ ]:
MAX_DATA_LEN = 40

random.seed(1)

data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과 : {Return_k_XOR_p.hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
    expected_ret = bytes(Return_k_XOR_p)
else:
    print('[✗] 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')


---

# ⚡ 5단계 — Lite 측 **전압 글리치** 모듈 설정 (★ 본 노트북 핵심 변경점 ★)

> **이 단계의 목표**
> Lite 의 글리치 모듈을 **전압 글리치(Crowbar VCC short)** 모드로 활성화하고,
> Lite 자체 ADC 를 *1 ADC 샘플 = 1 타겟 클럭* 으로 정렬해 글리치 시점(`ext_offset`) 과 샘플 인덱스를 1:1 로 매핑합니다.

---

### 5.1 클럭 글리치 vs 전압 글리치 — API 차이

| 함수 | 활성 글리치 | 출력 핀 | 영향 받는 신호 |
|:----|:----|:----|:----|
| `lite_scope.cglitch_setup()` | 클럭 글리치 (XOR 합성) | HS2 → CW308 CLKIN | 시스템 **클럭** |
| **`lite_scope.vglitch_setup()`** ★ 본 노트북 | **전압 글리치 (HP/LP MOSFET crowbar)** | **GLITCH SMA → CW308 GLITCH** | **타겟 VDD** |

`vglitch_setup()` 이 내부적으로 하는 일:

1. `scope.glitch.output = 'glitch_only'` 로 설정 — 글리치 펄스만 출력(HS2 XOR 비활성)
2. `scope.io.hs2 = 'clkgen'` 유지 — HS2 는 **깨끗한 클럭** 만 계속 출력
3. `scope.io.glitch_hp / glitch_lp` 는 우리가 명시적으로 제어 (어느 MOSFET 을 쓸지)
4. `scope.glitch.clk_src = 'clkgen'` 으로 글리치 펄스의 위상 기준을 시스템 클럭에 동기화

> 🔬 **`scope.io.glitch_hp` 와 `scope.io.glitch_lp`**
> - `glitch_hp = True` : **HP MOSFET 활성** — 큰 전류, 강한 단락, 일반 MCU 권장
> - `glitch_lp = True` : LP MOSFET 활성 — 미세 조정·저전력 타겟용
> - 둘 다 `True` 도 가능 (combined drive) — 강한 글리치가 필요한 경우
> - **둘 다 `False` 면 글리치가 전혀 안 들어갑니다** ← 자주 하는 실수


### 5.2 왜 여전히 1 sample = 1 clock 인가?

본 노트북에서도 *몇 번째 클럭에서 글리치가 발생하는지* 가 핵심 정보입니다.
Lite 의 ADC 는 션트 라인을 측정하지만, FIA 목적상 미세 누설보다 **클럭 단위 정밀 타이밍 매핑**이 더 중요합니다.

```
ADC 샘플레이트 = 타겟 클럭 × adc_mul × decimate_inverse

  · adc_mul = 1   → 1 클럭당 1 샘플 (FIA 시점 식별용)  ← 본 단계 (Lite)
  · adc_mul = 4   → 1 클럭당 4 샘플 (글리치 dip 형태 관찰용) ← 6단계 (Husky)
```

> 🔬 **두 ADC 의 비대칭 운용 (원본과 동일 패턴, 다른 의미)**
> - **Lite** `adc_mul = 1` → 클럭 단위 정밀 글리치 타이밍 (션트 측정)
> - **Husky** `adc_mul = 4` → **VDD dip 의 모양** 을 직접 보기 위한 4× 오버샘플링
>
> 클럭 글리치 판에서는 Husky 가 *글리치된 클럭 사각파* 를 보았지만, 본 전압 글리치 판에서는 **VDD 의 nanosecond 단위 dip** 을 봅니다. 이게 본 노트북의 결정적 산출물입니다.


In [ ]:
# ── 1) Lite 전압 글리치 모듈 활성화 ──────────────────
lite_scope.vglitch_setup('both')
# 'both' = HP + LP 둘 다 사용 가능 모드로 셋업 (개별 활성은 아래에서)
# 다른 옵션: 'hp', 'lp'  (그 MOSFET 만 사전 활성)

# ── 2) MOSFET 어느 쪽을 켤지 명시 ─────────────────────
# 권장: STM32F303 일반 타겟은 HP 단독, 또는 HP+LP
lite_scope.io.glitch_hp = True       # ★ HP MOSFET 활성
lite_scope.io.glitch_lp = False      #    LP 는 비활성 (미세 조정 시 True 로 추가)

# ── 3) 글리치 출력 모드 = 글리치 펄스만 ───────────────
lite_scope.glitch.output = 'glitch_only'
# 'clock_xor' 는 클럭 글리치 모드 — 본 노트북에서는 절대 사용 금지

# ── 4) 글리치 펄스의 위상 기준 = 시스템 클럭 ──────────
lite_scope.glitch.clk_src = 'clkgen'  # 시스템 클럭에 동기화된 위상 제어

# ── 5) Lite ADC: 1 샘플 = 1 클럭 (FIA 시점 매핑) ───
lite_scope.clock.adc_src    = 'clkgen_x1'
lite_scope.clock.clkgen_src = 'system'
lite_scope.clock.adc_mul    = 1
lite_scope.adc.decimate     = 1

# ── 6) 타겟 리셋 함수 (전압 글리치 환경에서는 더 자주 호출됨) ──
def reset_target_via_lite(scope):
    '''Lite 의 nRST 라인을 통해 타겟 MCU 리셋'''
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high_z'
    time.sleep(0.05)

reset_target_via_lite(lite_scope); time.sleep(1)
reset_target_via_lite(lite_scope); time.sleep(1)

# ── 7) 설정 검증 출력 ────────────────────────────────
print('=== Lite 전압 글리치 설정 ===')
print(f'  glitch.output       : {lite_scope.glitch.output}')
print(f'  glitch.clk_src      : {lite_scope.glitch.clk_src}')
print(f'  io.glitch_hp        : {lite_scope.io.glitch_hp}')
print(f'  io.glitch_lp        : {lite_scope.io.glitch_lp}')
print(f'  io.hs2 (clock out)  : {lite_scope.io.hs2}')
print()
print('=== Lite ADC / 클럭 설정 ===')
print(f'  clock.adc_src       : {lite_scope.clock.adc_src}')
print(f'  clock.clkgen_src    : {lite_scope.clock.clkgen_src}')
print(f'  clock.adc_mul       : {lite_scope.clock.adc_mul}')
print(f'  adc.decimate        : {lite_scope.adc.decimate}')
print()
print(f'  glitch.phase_shift_steps : {lite_scope.glitch.phase_shift_steps}')
print()
print('[✓] Lite: 전압 글리치 모드 + 1 ADC 샘플 = 1 타겟 클럭 정렬 완료')
print('[✓] Lite: GLITCH SMA 가 HP MOSFET drain 으로 활성화됨 (crowbar 준비)')
print()
print('⚠️ 이제 GLITCH SMA 가 활성 출력입니다.')
print('   글리치 실행 전 반드시 0.4.4 점퍼·0.5 디커플 캡 상태를 한 번 더 확인하세요.')


### 5.3 Lite `phase_shift_steps` 확인 + 글리치 파라미터 범위 산출 (전압 글리치 기준)

원본(클럭 글리치)에서는 `i_width` 가 클럭 1주기 내 위상 폭(%)이었습니다.
전압 글리치에서도 동일하게 `width` + `offset` 으로 펄스 모양을 정의하지만, **MOSFET 의 ON 시간 = 실제 VCC 단락 시간** 이므로 의미가 다릅니다.

| 파라미터 | 클럭 글리치에서의 의미 | **전압 글리치에서의 의미** |
|:----:|:----|:----|
| `offset` | 클럭 내 글리치 펄스 시작 위상 | MOSFET ON 시작 위상 (펄스 위상) |
| `width` | 클럭 내 글리치 펄스 폭 | **★ MOSFET ON 지속 시간** — 너무 크면 발열·brownout |
| `repeat` | 펄스 반복 수 | **★ 매우 신중히** — 5 이상은 MOSFET 가열·VDD 회복 실패 위험 |

> ⚠️ **전압 글리치 width 의 위험 영역**
> - 너무 작으면: 글리치 무효
> - 적정: 클럭 1주기의 ~30~50% (경험치)
> - 너무 크면: VDD 가 회복 못 함 → 타겟 brownout, 다시 부팅 못 함 → freezing
> - **`repeat × width` 가 클럭 주기를 초과하지 않도록** 8단계에서 가드를 둡니다.


In [ ]:
PS = lite_scope.glitch.phase_shift_steps

print(f'Lite phase_shift_steps : {PS}')
print(f'  · i_offset 허용 범위 : 0 ~ {PS}')
print(f'  · i_width  허용 범위 : 0 ~ {PS // 2}')
print(f'  · 1 step 당 위상   : 약 360°/ {PS} = {360.0/PS:.4f}° (이론치)')
print()
print('전압 글리치 권장 초기 범위 (8단계에서 정밀 탐색):')
print(f'  · i_offset 권장 : {int(PS*0.01)} ~ {int(PS*0.15)}  (= 1% ~ 15%)')
print(f'  · i_width  권장 : {int(PS*0.20)} ~ {int(PS*0.45)}  (= 20% ~ 45%)')


---

# 📡 6단계 — Husky 측 와이어태핑 환경 구성 (★VDD dip 직접 관찰★)

> **이 단계의 목표**
> Husky 가 **통신·글리치 제어에는 일체 개입하지 않으면서**, 트리거·클럭·**VDD 라인** 세 신호선을 외부에서 정확히 동기 측정하도록 구성합니다.
> 원본(클럭 글리치)판에서는 Husky 가 *글리치된 클럭* 을 봤지만, 본 노트북에서는 **VDD 의 nanosecond 단위 dip** 을 봅니다.

---

### 6.1 와이어태핑의 클럭 동기화 전략 (원본과 동일)

```
[1] PLL 입력 소스를 외부 AUX 로 전환
[2] AUX MCX 핀을 high-Z 입력 모드로 설정
[3] 내장 주파수 카운터로 외부 클럭 주파수 측정
[4] 측정 주파수의 최빈값으로 PLL 잠금
[5] ADC 클럭을 타겟 클럭의 4배로 정렬 (adc_mul=4)
[6] ADC 리셋 후 lock 상태 확인
```

> 💡 **본 노트북의 Husky 가 보는 클럭은 *깨끗한* 클럭입니다 (원본과의 결정적 차이)**
> 전압 글리치는 클럭 라인에는 손대지 않으므로, AUX MCX 로 들어오는 클럭은 항상 깨끗한 사각파입니다.
> 따라서 PLL 잠금이 원본(클럭 글리치 판)보다 훨씬 안정적이고, 글리치 활성 중에도 lock 이 풀릴 위험이 없습니다.


In [ ]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()
time.sleep(0.5)

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
husky_scope.clock.clkgen_freq = 0
husky_scope.clock.reset_adc()
# AUX MCX 를 입력(high-Z) 으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'
# PLL 입력 소스를 외부 클럭(extclk_aux_io) 으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
husky_scope.clock.reset_adc()

if (husky_scope.io.aux_io_mcx == 'high_z') and (husky_scope.clock.clkgen_src == 'extclk_aux_io'):
    print(f"[✓] io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")
    print(f"[✓] clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
else:
    print(f"[✗] 외부 클럭 설정 실패")


### 6.2 외부 클럭 주파수 탐색 + ADC 4× 오버샘플링

```python
husky_scope.clock.clkgen_freq = husky_scope.clock.freq_ctr
```

> 🔬 **본 노트북의 `adc_mul = 4` 가 중요한 이유**
> 전압 글리치 펄스의 폭은 보통 클럭 1주기의 20~45% (수십 nanosecond). 4× 오버샘플링이면 **dip 의 rising/falling edge 와 minimum depth 가 4개의 다른 샘플 값** 으로 표현되어 글리치 형태를 직접 측정할 수 있습니다.


In [ ]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
husky_scope.clock.pll._allow_rdiv = True
husky_scope.clock.freq_ctr_src = 'extclk'
time.sleep(0.5)

freqs = []
for _ in range(20):
    freqs.append(husky_scope.clock.freq_ctr)
    time.sleep(0.2)
data = pd.Series(freqs)
print(f"최빈값: {data.mode().iloc[0]} (등장 {(data == data.mode().iloc[0]).sum()}/{len(data)}회)")
print(f"범위:   {data.min()} ~ {data.max()} (Δ={data.max()-data.min()})")
print("\n[전체 통계 요약]")
print(data.describe())

husky_scope.clock.clkgen_freq = data.mode().iloc[0]
husky_scope.clock.adc_mul = 4
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("[✓] ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    print("[✗] ADC 클럭 동기화 실패 (Lock Error)")

if husky_scope.clock.clkgen_locked:
    print("[✓] Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    print("[✗] Husky PLL 잠금 실패! 외부 클럭의 진폭/듀티/안정성을 확인하세요.")


### 6.3 ★ VDD 와이어태핑용 ADC 입력 모드 설정 ★ (전압 글리치 핵심)

원본(클럭 글리치) 노트북에서는 Husky 의 MEAS 입력에 션트 양단이 연결되어 *상대적인 전류 변화* 를 측정했습니다.
본 노트북에서는 **VDD 라인을 직접 측정** 하므로 입력 모드와 게인이 달라집니다.

| 설정 | 원본 (Clock Glitch) | **본 노트북 (Voltage Glitch)** |
|:----:|:----:|:----|
| 측정 대상 | CW308 SHUNTL (션트 양단) | **타겟 VDD 핀 직결** |
| 신호 진폭 | mV 단위 (작은 변화) | **0V ~ 3.3V (큰 진폭)** |
| ADC 입력 모드 | 단일 종단(POS only) | 단일 종단 또는 **차동(`analog_diff`)** |
| 게인 권장 | 25 dB | **5 ~ 10 dB** (큰 진폭이므로 LNA 게인 낮춤) |
| `adc.offset` | 0 | 0 (트리거 직후) |

> ⚠️ **게인 25dB 로 두면 ADC 가 saturate 됩니다.**
> VDD 직결 측정 시 신호 진폭이 ~3.3V 수준이므로, LNA 를 통과시키면 ADC 입력 범위(약 ±1V) 를 크게 초과합니다.
> 본 노트북은 **게인 5dB** 를 기본으로 두고, 진폭 확인 후 조정합니다.


In [ ]:
# 트리거 입력 = 전면 USERIO D0 (CW308 의 GPIO4 / TRIG 를 분기해 입력) ⑦
husky_scope.trigger.triggers = 'userio_d0'
husky_scope.trigger.module   = 'basic'
husky_scope.adc.basic_mode   = 'rising_edge'

# 캡처 파라미터 (전압 글리치 — VDD 직결 측정용)
husky_scope.gain.db        = 5       # ★ LNA 게인 낮춤 (큰 진폭 대응)
husky_scope.adc.samples    = 5000
husky_scope.adc.offset     = 0
husky_scope.adc.presamples = 0

# (옵션) 차동 입력 모드 — 더 깨끗한 글리치 파형을 원할 때
# husky_scope.adc.input = 'analog_diff'   # 주석 해제 시 NEG 측에 기준 전압 필요

print(f"[✓] Husky 스코프 캡처 파라미터 설정 완료 (VDD 와이어태핑 모드)")
print(f"  trigger.triggers : {husky_scope.trigger.triggers}")
print(f"  trigger.module   : {husky_scope.trigger.module}")
print(f"  adc.basic_mode   : {husky_scope.adc.basic_mode}")
print(f"  gain.db          : {husky_scope.gain.db}  (★ VDD 직결 측정용 낮은 게인)")
print(f"  adc.samples      : {husky_scope.adc.samples}")
print(f"  adc.offset       : {husky_scope.adc.offset}")
print()
print('💡 첫 베이스라인 캡처 후 파형 진폭을 확인하고 필요시 게인 조정 (3~10dB 범위)')


---

# 🌊 7단계 — 베이스라인 (글리치 없음) 동시 캡처 + 글리치 시점 후보 식별

> **이 단계의 목표**
> 글리치를 *주입하지 않은 정상 상태* 의 파형을 Lite·Husky 두 스코프로 동시 수집해
> (a) 정상 출력값 `expected_ret` 를 재확인,
> (b) Lite 의 `trig_count` 로 *연산 클럭 수* 측정,
> (c) Husky 의 **순수 VDD 파형** 으로 *연산 구간을 시각적 식별*.

---

### 7.1 글리치 비활성 모드 (`arm_timing = 'no_glitch'`)

| `arm_timing` 값 | 의미 |
|:----:|:----|
| `'after_scope'` | scope arm 직후부터 글리치 활성 (8단계 탐색용) |
| `'no_glitch'`   | **글리치 비활성** (베이스라인 수집용 / 디버그용) |

> 🛡 **전압 글리치 안전 모드**
> `'no_glitch'` 로 두면 MOSFET 이 OFF 상태로 유지되어 VDD 가 안정된 3.3V 를 유지합니다. **베이스라인 캡처 시 반드시 이 모드** 를 사용해야 합니다.


In [ ]:
# Lite 측: 글리치 비활성 + glitch_only 출력 모드 유지
lite_scope.glitch.arm_timing = 'no_glitch'
lite_scope.glitch.output     = 'glitch_only'  # ← 클럭 글리치 판과 다른 점

# 타겟 초기화 + 데이터 주입
reset_target_via_lite(lite_scope)
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# Lite ADC 샘플 수 임시 충분값 (첫 캡처 후 trig_count 로 재설정)
lite_scope.adc.samples = 24400

# 양쪽 스코프 모두 arm → 캡처 → 트레이스 회수
target.flush()
husky_scope.arm()
lite_scope.arm()
target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
ack = target.read_cmd(timeout=500)

ret_husky = husky_scope.capture()
ret_lite  = lite_scope.capture()
if ret_husky or ret_lite:
    raise RuntimeError(f"베이스라인 캡처 실패: husky={ret_husky}, lite={ret_lite} — 배선·트리거 점검 필요")

target.flush()
target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
ret_payload = target.read_cmd(timeout=500)

trace_lite_baseline  = np.array(lite_scope.get_last_trace())
trace_husky_baseline = np.array(husky_scope.get_last_trace())

expected_ret_now = ret_payload[3 : 3 + ret_payload[2]]
assert expected_ret_now == expected_ret, "베이스라인 출력이 4단계 골든값과 다릅니다!"

trig_count_baseline = int(lite_scope.adc.trig_count)
lite_scope.adc.samples = trig_count_baseline + 50

# Husky VDD 베이스라인의 평균값 — 게인 조정 가이드
vdd_mean = float(np.mean(trace_husky_baseline))
vdd_peak = float(np.max(trace_husky_baseline) - np.min(trace_husky_baseline))

print(f'\n=== 베이스라인 ===')
print(f'  Lite  파형 길이      : {len(trace_lite_baseline)} 샘플  (≈ {len(trace_lite_baseline)} clock)')
print(f'  Husky 파형 길이      : {len(trace_husky_baseline)} 샘플  (≈ {len(trace_husky_baseline)//4} clock)')
print(f'  Lite trig_count     : {trig_count_baseline}')
print(f'  → 다음 단계용 samples : {lite_scope.adc.samples}')
print(f'  expected_ret        : {expected_ret.hex(" ")}')
print()
print(f'=== Husky VDD 베이스라인 진폭 분석 ===')
print(f'  평균 전압 (정규화)  : {vdd_mean:+.4f}')
print(f'  Peak-to-peak       : {vdd_peak:.4f}')
print()
if vdd_peak > 0.9:
    print('  ⚠️ 진폭이 큽니다 — 게인을 더 낮추세요 (gain.db = 0 ~ 3 시도)')
elif vdd_peak < 0.05:
    print('  ⚠️ 진폭이 너무 작습니다 — 게인을 높이거나 VDD 직결 배선을 확인하세요')
else:
    print('  [✓] VDD 와이어태핑 진폭이 적정 범위 — 게인 유지')

print(f'\n[✓] 베이스라인 동시 캡처 성공 — Husky 가 동일 트리거에 정렬 측정됨')
print(f'    (전압 글리치 비활성 상태이므로 Husky 파형은 정상 동작 중 VDD 패턴을 보여줍니다)')


### 7.2 베이스라인 파형 시각화 — Lite 션트 측정 vs Husky VDD 직결

두 파형을 비교해 확인할 것:
- **Lite 션트 측정 (1 clk/sample)** : XOR 루프 반복 패턴 (170 cycles 부근)
- **Husky VDD 직결 (4 sample/clock)** : 정상 동작 시 VDD 는 **거의 평탄** 한 ~3.3V 부근 — 글리치가 없으니 dip 이 없어야 함

> 🔬 **이 그래프에서 식별할 것**
> - Husky 파형이 **평탄한 라인** 이라면 VDD 와이어태핑 정상
> - Husky 파형에 *주기적 ripple* 이 보인다면 → 디커플 캡을 너무 많이 제거했거나 전원이 불안정 → 0.5절로 복귀
> - Lite 파형의 for-loop 패턴이 보이는 구간 (트리거 후 약 30 ~ 200 clk) 이 8단계 `ext_offset` 탐색 범위


In [ ]:
# ── Bokeh: Lite 베이스라인 (1 sample = 1 clock) ────────
p1 = figure(
    width=900, height=280,
    title='Baseline — Lite (shunt, 1 sample = 1 clock, no glitch)',
    x_axis_label='Sample Index (= Clock Cycle from Trigger)',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)
p1.line(np.arange(len(trace_lite_baseline)), trace_lite_baseline,
        line_width=1.2, line_color='#2E86AB')
p1.title.text_font_size = '12pt'
p1.grid.grid_line_alpha = 0.3
p1.outline_line_color = None
p1.add_tools(HoverTool(tooltips=[('Clock', '@x{0}'), ('V', '@y{0.0000}')], mode='vline'))

# ── Bokeh: Husky 베이스라인 (VDD 직결, 4 samples/clock) ─
husky_x = np.arange(len(trace_husky_baseline)) / 4.0
p2 = figure(
    width=900, height=280,
    title='Baseline — Husky (★VDD direct wire-tap★, 4× oversample, NO GLITCH)',
    x_axis_label='Clock Cycle from Trigger  (= sample / 4)',
    y_axis_label='VDD (normalized)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)
p2.line(husky_x, trace_husky_baseline, line_width=1.0, line_color='#A23B72', line_alpha=0.85)
p2.title.text_font_size = '12pt'
p2.grid.grid_line_alpha = 0.3
p2.outline_line_color = None
p2.add_tools(HoverTool(tooltips=[('Clock', '@x{0.00}'), ('V', '@y{0.0000}')], mode='vline'))

show(column(p1, p2))


---

# 🎯 8단계 — 전압 글리치 파라미터 탐색 + Husky VDD 와이어태핑 동시 측정

> **이 단계의 목표**
> `(ext_offset, offset, width)` 의 3차원 파라미터 공간을 격자 탐색하며,
> 각 시행마다 **Lite 션트 + Husky VDD** 로 동시 캡처해, 결과 6단계 분류와 함께 **결과 코드별 VDD dip 파형**을 보존합니다.

---

### 8.1 전압 글리치 파라미터 범위 — 클럭 글리치와 다른 점

| 항목 | 클럭 글리치 (원본) | **전압 글리치 (본 노트북)** |
|:----:|:----:|:----|
| `i_offset` 권장 | 1 ~ 10% | **1 ~ 15%** (펄스 시작 위상, 비슷한 범위) |
| `i_width` 권장 | 30 ~ 40% | **20 ~ 45%** (MOSFET ON 시간, 더 넓은 탐색 권장) |
| `repeat` | 보통 1 | **1 ~ 3 (★ 4 이상 금지)** |
| 코드 0(freezing) 비율 | 보통 ~10% | **종종 30~50%** — VDD brownout 빈발 |
| 회복 전략 | 글리치 종료 시 즉시 정상 | **★ 매 시행 후 충분한 nRST + 휴식** |

### 8.2 ★ MOSFET 보호 — 본 노트북 고유 안전 로직

전체 탐색이 1000+ 시행에 이르므로, 누적 MOSFET 발열을 관리합니다.

```
- 글리치 시행마다 짧은 sleep (50ms) 으로 MOSFET 휴식
- 100 시행마다 5초 휴식
- `i_width × repeat > PS/2` 인 조합은 자동 스킵 (과도 펄스 방지)
- freezing 연속 발생 시 자동 nRST 강화 (50ms → 200ms)
```

### 8.3 arm 순서·캡처 검사 (원본과 동일)

| 순서 | 동작 | 의미 |
|:----:|:----|:----|
| 1 | `husky_scope.arm()` | Husky 가 트리거 대기 |
| 2 | `lite_scope.arm()` | Lite 가 트리거 대기 + 글리치 준비 |
| 3 | `target.send_cmd(0x82, ...)` | 펌웨어가 GPIO4 토글 → 두 스코프 동시 트리거 |
| 4 | 양쪽 `capture()` 검사 | 하나라도 실패 = freezing 추정 |


In [ ]:
# ── 글리치 발생 시점 + 출력 모드 설정 (탐색 활성) ─────
lite_scope.glitch.arm_timing = 'after_scope'   # scope arm 후 글리치 활성
lite_scope.glitch.output     = 'glitch_only'   # 전압 글리치
lite_scope.glitch.repeat     = 1               # ★ MOSFET 보호 위해 1 로 고정

# ── 경고 로그 무시 (탐색 중 빈번하게 발생하는 정상 경고) ──
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)

# ── 탐색 범위 (PS 기반 상대비율, 전압 글리치 권장값) ───
PS = lite_scope.glitch.phase_shift_steps

i_ext_offsets = list(range(170, 183))   # baseline trig_count 부근
i_offsets_pct = [0.01, 0.03, 0.05, 0.08, 0.12]
i_widths_pct  = [0.20, 0.28, 0.34, 0.40]     # ★ 클럭 글리치보다 넓은 탐색

i_offsets = sorted(set(int(PS * f) for f in i_offsets_pct))
i_widths  = sorted(set(int(PS * f) for f in i_widths_pct))

# 안전 가드: width 가 너무 크면 제외
i_widths = [w for w in i_widths if w <= PS // 2]

REPS_PER_PARAM = 5

print(f'phase_shift_steps (Lite) : {PS}')
print(f'i_ext_offset 탐색 범위    : {i_ext_offsets[0]} ~ {i_ext_offsets[-1]}  ({len(i_ext_offsets)} 개)')
print(f'i_offset     후보         : {i_offsets}')
print(f'i_width      후보         : {i_widths}')
print(f'glitch.repeat            : {lite_scope.glitch.repeat}  (★ MOSFET 보호)')
print(f'총 시행 횟수             : {len(i_ext_offsets) * len(i_offsets) * len(i_widths) * REPS_PER_PARAM}')

print('\n[✓] 전압 글리치 출력 모드 활성화 완료 — arm_timing=after_scope')
print('⚠️ 이제부터 GLITCH SMA 가 능동 출력입니다. 보드에 손대지 마세요.')


### 8.4 결과 저장 자료구조 (원본과 동일)

```
cglitch_result      [코드, ext_offset, width, offset]
cglitch_extra_data  [expected_ret, actual_ret]
husky_by_code, lite_by_code  결과 코드별 파형 (각 코드당 최대 MAX_TRACES_PER_CODE 개)
```


In [ ]:
# 결과 컨테이너 (원본과 동일 구조)
cglitch_result     = []   # [코드, ext_offset, width, offset]
cglitch_extra_data = []   # [expected_ret, actual_ret]

MAX_TRACES_PER_CODE = 20

husky_by_code = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}
lite_by_code  = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}
ctx_by_code   = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}

# 전압 글리치 전용 안전 카운터
freeze_streak = 0          # 연속 freezing 카운트
total_freezes = 0
trials_done   = 0

print('자료구조 초기화 완료')


### 8.5 메인 탐색 루프 (전압 글리치 안전 보호 + Husky VDD 동시 캡처)

```
for each (ext_offset, offset, width):
    for each 반복 (5회):
        1. Lite 글리치 파라미터 설정
        2. 타겟 리셋 + (k, p, l) 주입
        3. ★ husky_scope.arm()  →  lite_scope.arm()
        4. 0x82 'c' 송신 → 타겟 GPIO4 트리거 → 두 스코프 동시 캡처
        5. 두 스코프 capture() 검사 (freezing 판정)
        6. 양쪽 트레이스 회수
        7. 결과 회수 (0x83 'r')
        8. 6단계 분류
        9. ★ freezing 누적 시 nRST 강화 + 휴식
       10. ★ 100 시행마다 5초 휴식 (MOSFET 보호)
```

> ⏱ **실행 시간 / 안전 안내**
> - 약 1300 시행 × 0.35초 ≈ **8분**, 단 freezing 비율에 따라 12분까지 늘어남
> - **연속 freezing 5회 이상** 발생 시 셀이 자동 일시정지 → 0단계 점검 권장 메시지
> - 셀 실행 중 보드 표면 (특히 STM32 칩 주변) **40℃ 이상** 이면 즉시 셀 중단(상단 ◼)


In [ ]:
# ══════════════════════════════════════════════════════════
#  메인 파라미터 탐색 루프 — Lite voltage glitch + Husky VDD wire-tap
# ══════════════════════════════════════════════════════════

REST_EVERY_N        = 100   # N 시행마다 MOSFET 휴식
REST_SECONDS        = 5
FREEZE_STREAK_LIMIT = 5     # 연속 freezing 임계
STRONG_RESET_HOLD   = 0.20  # freezing 연속 시 nRST 유지 시간 (초)

def strong_reset(scope):
    '''freezing 연속 시 더 강한 리셋'''
    scope.io.nrst = 'low'
    time.sleep(STRONG_RESET_HOLD)
    scope.io.nrst = 'high_z'
    time.sleep(STRONG_RESET_HOLD)

for i_ext_offset in trange(i_ext_offsets[0], i_ext_offsets[-1] + 1,
                           desc='i_ext_offset', leave=False):

    for i_offset in i_offsets:
        for i_width in i_widths:

            # 유효성 검사 + 안전 가드
            if i_width == 0:
                continue
            if (i_offset + i_width) > PS:
                continue
            # ★ 전압 글리치 추가 가드: width × repeat 이 너무 크면 스킵
            if (i_width * lite_scope.glitch.repeat) > (PS // 2):
                continue

            for _ in range(REPS_PER_PARAM):

                trials_done += 1

                # 1. Lite 글리치 파라미터 설정
                lite_scope.glitch.ext_offset = i_ext_offset
                lite_scope.glitch.offset     = i_offset
                lite_scope.glitch.width      = i_width

                # 2. 타겟 초기화 (freezing 연속 시 강한 리셋)
                if freeze_streak >= FREEZE_STREAK_LIMIT:
                    strong_reset(lite_scope)
                    freeze_streak = 0
                else:
                    reset_target_via_lite(lite_scope)

                my_fsr_cmd(target, 0x81, 'k', data_k)
                my_fsr_cmd(target, 0x81, 'p', data_p)
                my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

                # 3. 두 스코프 동시 arm (Husky → Lite 순)
                target.flush()
                husky_scope.arm()
                lite_scope.arm()

                # 4. 연산 트리거
                target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
                actual_ret = target.read_cmd(timeout=500)

                # 5. 캡처 검사 + freezing 판정
                cap_husky = husky_scope.capture()
                cap_lite  = lite_scope.capture()
                if (actual_ret is None) or cap_lite:
                    code_now = 0
                    freeze_streak += 1
                    total_freezes += 1
                    cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    if (not cap_husky) and len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                        husky_by_code[code_now].append(np.array(husky_scope.get_last_trace()))
                        lite_by_code[code_now].append(np.array(lite_scope.get_last_trace()) if not cap_lite else None)
                        ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))
                    # MOSFET 보호 휴식
                    time.sleep(0.05)
                    continue

                # 6. 두 스코프 트레이스 회수
                trace_husky = np.array(husky_scope.get_last_trace())
                trace_lite  = np.array(lite_scope.get_last_trace())

                # 7. 결과 페이로드 회수
                target.flush()
                target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
                actual_ret = target.read_cmd(timeout=500)
                if actual_ret is None:
                    code_now = 0
                    freeze_streak += 1
                    total_freezes += 1
                    cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    if len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                        husky_by_code[code_now].append(trace_husky)
                        lite_by_code[code_now].append(trace_lite)
                        ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))
                    time.sleep(0.05)
                    continue
                actual_ret = actual_ret[3 : 3 + actual_ret[2]]
                freeze_streak = 0  # 정상 응답 → 연속 freeze 초기화

                # 8. 6단계 분류
                if expected_ret == actual_ret:
                    code_now = 1
                elif lite_scope.adc.trig_count < (lite_scope.adc.samples - 30):
                    code_now = 2
                else:
                    n_diff = sum(g != r for g, r in zip(expected_ret, actual_ret))
                    if   n_diff == 1: code_now = 3
                    elif n_diff < 5:  code_now = 4
                    else:             code_now = 5

                cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                cglitch_extra_data.append([expected_ret, actual_ret])

                # 9. 결과 코드별 husky/lite 파형 보존
                if len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                    husky_by_code[code_now].append(trace_husky)
                    lite_by_code[code_now].append(trace_lite)
                    ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))

                if code_now in (2, 3):
                    LABEL = {2: 'loop skip', 3: '1-byte fault'}[code_now]
                    print(f'\n[코드{code_now}] {LABEL}  '
                          f'ext={i_ext_offset}  off={i_offset}  w={i_width}  '
                          f'ret_diff_bytes={sum(g!=r for g,r in zip(expected_ret, actual_ret))}')

                # MOSFET 짧은 휴식
                time.sleep(0.02)

                # 10. 100 시행마다 MOSFET 휴식
                if trials_done % REST_EVERY_N == 0:
                    print(f'\n[휴식] 누적 {trials_done} 시행 — MOSFET 보호 {REST_SECONDS}s 휴식  (freezes: {total_freezes})')
                    # 글리치 비활성 후 휴식
                    lite_scope.glitch.arm_timing = 'no_glitch'
                    time.sleep(REST_SECONDS)
                    lite_scope.glitch.arm_timing = 'after_scope'

print('\n[✓] 전압 글리치 파라미터 탐색 완료')
print(f'  총 시행 수    : {len(cglitch_result)}')
print(f'  freezing 수   : {total_freezes}  ({100*total_freezes/max(1,len(cglitch_result)):.1f}%)')

# 안전 종료: 글리치 비활성으로 복귀
lite_scope.glitch.arm_timing = 'no_glitch'
print('[✓] 글리치 비활성 모드로 복귀 — MOSFET OFF')


---

# 📊 9단계 — 통계 분석으로 최적 글리치 파라미터 도출

> **이 단계의 목표**
> 결과 코드별 빈도와 *최빈* `(offset, width)` 를 산출합니다.
> 원본 노트북과 분석 로직은 동일하나, **전압 글리치에서는 코드 0(freezing) 비율이 상대적으로 높을 수 있으니** 그 점을 감안해 해석합니다.

---


In [ ]:
cglitch_result_arr = np.array(cglitch_result, dtype=np.float64)

print(f'전체 시행 횟수 : {len(cglitch_result_arr)}')
print(f'배열 shape    : {cglitch_result_arr.shape}  (rows, [code, ext_offset, width, offset])')


In [ ]:
LABELS = {
    0: 'freezing',
    1: 'normal',
    2: 'for-loop skip',
    3: 'one faulty byte',
    4: 'few faulty bytes (<5)',
    5: 'etc (>=5 faults)',
}

print('=' * 64)
print(f'{"코드":>5} {"분류":<24} {"건수":>8} {"best offset":>14} {"best width":>10}')
print('=' * 64)

for code_, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code_
    cnt  = int(mask.sum())

    if cnt == 0:
        print(f'{code_:>5} {label:<24} {cnt:>8} {"—":>14} {"—":>10}')
        continue

    best_offset = sp.stats.mode(cglitch_result_arr[mask, 3], keepdims=False).mode
    best_width  = sp.stats.mode(cglitch_result_arr[mask, 2], keepdims=False).mode
    print(f'{code_:>5} {label:<24} {cnt:>8} {int(best_offset):>14} {int(best_width):>10}')

print('=' * 64)


> 💡 **결과 해석 가이드 (전압 글리치 판)**
>
> - **코드 2 (for-loop skip)** 의 `best (offset, width)` → **인증 우회 공격용** 후보 파라미터
> - **코드 3 (1-byte fault)** 의 `best (offset, width)` → **DFA 키 복원용** 후보 파라미터
> - 코드 0(freezing) 비율이 **>50%** 면 → 글리치가 너무 강함 → `i_width` 줄이거나 LP 만 사용
> - 코드 1(normal) 비율이 **>80%** 면 → 글리치가 너무 약함 → `i_width` 늘리거나 디커플 캡 추가 제거
> - 코드 0 이 *특정 width* 구간에서만 집중 → 그 구간이 **brownout threshold** — 그 *직전* width 가 sweet spot
>
> ★ **본 노트북의 추가 산출물**: Husky 의 **VDD dip 파형 모양** 으로 *코드별 글리치 특성* 을 시각화 (10단계).


---

# 🎨 10단계 — 시각화: 파라미터 분포 + ★VDD dip 파형 비교★

> **이 단계의 목표**
> 본 노트북의 핵심 산출물 3가지를 인터랙티브 그래프로 제시:
>
> 1. **(offset, width) 평면 분포**
> 2. ★ **결과 코드별 Husky VDD dip 파형 비교** ← 전압 글리치 고유 산출물
> 3. ★ **Lite 션트 vs Husky VDD 비교** ← 측정 경로 신뢰성 검증

---

### 10.1 (offset, width) 평면 산점도


In [ ]:
COLOR_MAP = {
    0: '#888888',   # freezing      — gray
    1: '#1f77b4',   # normal        — blue
    2: '#2ca02c',   # loop skip     — green ✅
    3: '#d62728',   # 1-byte fault  — red ✅
    4: '#ff7f0e',   # few faults    — orange
    5: '#9467bd',   # etc           — purple
}

p_map = figure(
    width=900, height=500,
    title='Voltage Glitch Parameter Map — (offset, width) plane  [Lite PS-relative]',
    x_axis_label='i_offset (1-clock 내 시작 위상, Lite phase_shift_steps 기준)',
    y_axis_label='i_width  (MOSFET ON 폭, Lite phase_shift_steps 기준)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)

for code_, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code_
    if not mask.any():
        continue
    src = ColumnDataSource(data=dict(
        x = cglitch_result_arr[mask, 3],
        y = cglitch_result_arr[mask, 2],
        ext = cglitch_result_arr[mask, 1],
        label = [label] * int(mask.sum()),
    ))
    p_map.scatter('x', 'y', source=src,
                  size=8, alpha=0.55,
                  color=COLOR_MAP[code_],
                  legend_label=f'[{code_}] {label} (n={int(mask.sum())})')

p_map.title.text_font_size = '13pt'
p_map.title.align = 'center'
p_map.grid.grid_line_alpha = 0.3
p_map.outline_line_color = None
p_map.legend.location = 'top_right'
p_map.legend.click_policy = 'hide'
p_map.legend.label_text_font_size = '9pt'
p_map.legend.background_fill_alpha = 0.85
p_map.add_tools(HoverTool(tooltips=[
    ('result', '@label'),
    ('i_offset', '@x{0}'),
    ('i_width',  '@y{0}'),
    ('i_ext_offset', '@ext{0}'),
]))

show(p_map)


### 10.2 ★ 결과 코드별 Husky VDD dip 파형 비교 ★ (전압 글리치 고유 산출물)

각 결과 코드별로 보존된 Husky **VDD 와이어태핑** 파형을 겹쳐 그립니다.

| 비교 쌍 | 의미 |
|:----:|:----|
| 코드 1 vs 코드 2 | **dip 의 *위치(time)*** 가 어떻게 다른가? — 루프 종료 시점 부근에서 dip 이 발생하는가? |
| 코드 1 vs 코드 3 | dip 의 **깊이·폭** 차이 — 단일 instruction 변조에 필요한 최소 dip 강도는? |
| 코드 0 (freezing) | dip 이 너무 깊어 VDD 가 회복 못 함 — **brownout threshold** 식별 |

> 🔬 **본 노트북의 가장 강력한 산출물**
> 단일 장치 FIA 에서는 "내가 의도한 width" 만 알 수 있지만, **Husky 가 측정한 실제 dip 깊이·폭** 을 *결과 코드별로 비교* 하면 다음을 도출할 수 있습니다:
> - **최소 fault 유발 dip 깊이 (mV)** — 디바이스 의존 보안 파라미터
> - **dip rise/fall time 의 영향** — 디커플 잔량 추정
> - **fault category 별 *물리적* 임계값** — 후속 차세대 공격(SIFA) 설계의 기반


In [ ]:
# 결과 코드별 husky VDD 파형 겹쳐 그리기
panels = []
for code_, label in LABELS.items():
    traces = husky_by_code.get(code_, [])
    if len(traces) == 0:
        continue

    # None 트레이스(freezing 중 husky 만 캡처된 경우) 제거
    traces_clean = [t for t in traces if t is not None and len(t) > 0]
    if len(traces_clean) == 0:
        continue

    arr = np.array(traces_clean)
    n_show = min(len(traces_clean), MAX_TRACES_PER_CODE)
    sample_axis_clk = np.arange(arr.shape[1]) / 4.0

    p = figure(
        width=900, height=240,
        title=f'★ Husky VDD wire-tap ★ — Code {code_} : {label}  (n={n_show})',
        x_axis_label='Clock Cycle from Trigger',
        y_axis_label='VDD (normalized)',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    for i in range(n_show):
        p.line(sample_axis_clk, arr[i],
               line_width=1.0,
               line_color=COLOR_MAP[code_],
               line_alpha=0.40)
    mean_trace = arr[:n_show].mean(axis=0)
    p.line(sample_axis_clk, mean_trace,
           line_width=2.0, line_color='#1a1a1a', line_alpha=0.95,
           legend_label='mean VDD trace')
    p.title.text_font_size = '11pt'
    p.grid.grid_line_alpha = 0.3
    p.outline_line_color = None
    p.legend.location = 'top_right'
    p.legend.label_text_font_size = '9pt'
    panels.append(p)

if panels:
    show(column(*panels))
else:
    print('보존된 husky 파형이 없습니다. 8단계가 정상 실행되었는지 확인하세요.')


### 10.3 동일 시행에서의 Lite 션트 측정 vs Husky VDD 직결 비교

가장 흥미로운 결과 코드 (코드 2 또는 3) 에서 **첫 번째 보존 시행** 을 골라 양쪽 파형을 비교합니다.

> 🔬 **이 비교에서 점검할 것**
> - **빨간 점선(글리치 발생 시점)** 좌우의 파형 변화
> - Lite 션트 측정: dip *순간* 의 션트 양단 전압 변화 → 큰 current spike 가 보일 수 있음
> - Husky VDD: 실제 dip 의 깊이·폭 직접 관찰
> - 두 측정이 *동일 시점* 에 일관된 신호를 보인다면 두 와이어태핑 모두 신뢰 가능


In [ ]:
candidate_code = None
for c_ in (2, 3, 4, 5, 0):
    if len(husky_by_code[c_]) > 0:
        candidate_code = c_
        break

if candidate_code is None:
    print('비교에 사용할 보존 시행이 없습니다.')
else:
    husky_t = husky_by_code[candidate_code][0]
    lite_t  = lite_by_code [candidate_code][0]
    ctx     = ctx_by_code  [candidate_code][0]
    label   = LABELS[candidate_code]

    print(f'비교 대상: Code {candidate_code} ({label})')
    print(f'  ext_offset = {ctx[0]}  /  i_offset = {ctx[1]}  /  i_width = {ctx[2]}')

    # Lite (1 sample = 1 clock) — 션트 측정
    p_lite = figure(
        width=900, height=260,
        title=f'Lite shunt measure (1 sample = 1 clock) — Code {candidate_code} : {label}',
        x_axis_label='Sample (= Clock from Trigger)',
        y_axis_label='V',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    if lite_t is not None:
        p_lite.line(np.arange(len(lite_t)), lite_t, line_width=1.2, line_color='#2E86AB')
    p_lite.grid.grid_line_alpha = 0.3
    p_lite.outline_line_color = None
    p_lite.add_layout(Span(location=ctx[0], dimension='height',
                           line_color='#d62728', line_dash='dashed', line_width=1.5))

    # Husky (4 samples / clock) — VDD 직결
    husky_x_clk = np.arange(len(husky_t)) / 4.0
    p_husky = figure(
        width=900, height=260,
        title=f'★ Husky VDD direct wire-tap ★ (4× oversample) — Code {candidate_code} : {label}',
        x_axis_label='Clock Cycle from Trigger',
        y_axis_label='VDD (normalized)',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    p_husky.line(husky_x_clk, husky_t, line_width=1.0, line_color='#A23B72', line_alpha=0.9)
    p_husky.grid.grid_line_alpha = 0.3
    p_husky.outline_line_color = None
    p_husky.add_layout(Span(location=ctx[0], dimension='height',
                            line_color='#d62728', line_dash='dashed', line_width=1.5))

    show(column(p_lite, p_husky))


> 💡 **두 파형 비교에서 점검할 사항**
>
> - **빨간 점선 = 글리치 발생 시점 (`ext_offset`)** : 두 그래프에서 동일한 x 좌표
> - **VDD 와이어태핑 (Husky)** : `ext_offset` 부근에서 **VDD 가 nanosecond 단위로 GND 까지 dip** 하는 모습이 관찰되어야 함 (전압 글리치의 *결정적 시각적 증거*)
> - **션트 측정 (Lite)** : dip 순간 큰 current spike 또는 평소와 다른 패턴
> - 두 파형의 거시 패턴이 일치한다면 와이어태핑 경로가 신뢰할 만함을 시각적으로 입증


---

# 🔚 마무리 — 안전한 다중 장치 자원 해제 (★ MOSFET 안전 종료 우선 ★)

> **이 단계의 목표**
> 노트북 종료 전 **MOSFET 비활성화 → target 해제 → scope 해제** 순서로 자원을 정리합니다.

---

### 전압 글리치 전용 종료 순서 (원본과 다른 점)

1. **★ Lite 글리치 비활성화 (MOSFET OFF)** — `glitch.arm_timing = 'no_glitch'` + `io.glitch_hp/lp = False`
2. `target.dis()` — Lite UART 채널 해제
3. 각 `scope.dis()` — Husky, Lite 순으로 USB 핸들 해제

> ⚠️ **MOSFET 을 먼저 OFF 해야 하는 이유**
> 만약 `glitch_hp = True` 상태로 `scope.dis()` 를 호출하면, 일부 리비전에서는 그 상태가 유지되어 다음 세션 시작 시 곧바로 단락이 발생할 수 있습니다. 명시적으로 끄는 것이 안전합니다.


In [ ]:
def disconnect_all_devices_safely(scopes: dict) -> None:
    # 0. ★ MOSFET 명시적 비활성 — 전압 글리치 전용
    try:
        lite_scope.glitch.arm_timing = 'no_glitch'
        lite_scope.io.glitch_hp = False
        lite_scope.io.glitch_lp = False
        print("  [✓] Lite MOSFET OFF (전압 글리치 비활성화)")
    except Exception as e:
        print(f"  [!] MOSFET OFF 실패  └─ {e}")

    # 1. 타겟 객체 닫기 (Lite UART 점유 해제)
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")

    # 2. 각 scope 해제
    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
    scopes.clear()

# 파형 수집 및 데이터 저장 완료 후 반드시 자원 반환
disconnect_all_devices_safely(scopes)

print('\n[ℹ️ 물리적 후속 작업]')
print('  · USB 분리 전 보드 표면 온도 확인 (만지기 어려운 정도면 5분 더 방치)')
print('  · GLITCH SMA 케이블이 능동 출력이었으므로, 분리 시 SMA shell 부터 먼저 풀기')
print('  · 디커플 캡을 영구 제거했다면 보드 라벨/문서에 표시')


---

## 📝 본 노트북 요약 (전압 글리치 고도화 판)

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 0 | **하드웨어 점검 (점퍼·디커플·MOSFET·VDD 배선)** | 검증된 물리 셋업 |
| 1 | `cw.list_devices()` + `cw.scope(sn=...)` | 다중 장치 동시 연결 |
| 2 | `cw.target(lite_scope, SimpleSerial2)` | Lite ↔ 타겟 통신 채널 |
| 3 | `make` + `cw.program_target(lite_scope, ...)` | Lite 가 프로그래머 |
| 4 | `my_fsr_cmd()` + Golden Model | 통신·연산 정상성 검증 |
| 5 | **`lite_scope.vglitch_setup('both')` + `io.glitch_hp = True`** | **★ Lite MOSFET crowbar 활성** |
| 6 | `extclk_aux_io` + **`gain.db = 5`** + `adc_mul=4` | **★ Husky 가 VDD 직결 와이어태핑** |
| 7 | `arm_timing='no_glitch'` + 동시 캡처 | 베이스라인 파형 |
| 8 | 두 스코프 동시 arm + **MOSFET 보호 휴식 로직** | 결과 코드별 husky/lite 파형 |
| 9 | `np.array` + `sp.stats.mode` | 결과 코드별 최빈 `(offset, width)` |
| 10 | Bokeh: 산점도 + ★**VDD dip 코드별 비교**★ + Lite/Husky 비교 | 인터랙티브 시각화 |
| 마무리 | **MOSFET OFF → target.dis() → scope.dis()** | 안전 종료 |

### ✅ 본 노트북에서 새로 익혀야 할 핵심 개념 (전압 글리치 고유)

1. **Crowbar VCC short** — HP/LP MOSFET 이 VDD 를 nanosecond 단위로 GND 단락
2. **하드웨어 개조의 결정성** — SJ4(OPEN)/SJ5(CLOSED) + 디커플 커패시터 제거 = 성패의 90%
3. **VDD 직접 와이어태핑** — Husky MEAS POS 에 *타겟 VDD 핀 직결* → glitch dip 모양 직접 관찰
4. **MOSFET 발열·brownout 관리** — `repeat=1`, width 가드, 100 시행마다 휴식
5. **두 신호 경로의 분리** — HS2(깨끗한 클럭) vs GLITCH SMA(crowbar) — 클럭 글리치 판과 가장 큰 차이
6. **VDD dip 의 *결과 코드별 형태 차이*** — 단일 장치 FIA 에서는 보이지 않는 물리적 fault 시그니처

### 🔬 후속 연구 방향 제안 (전압 글리치 특화)

- **dip 깊이 정량화** — Husky VDD 파형에서 *최소 dip 깊이 × fault 유발률* 곡선 도출 → 디바이스별 brownout threshold 데이터
- **fault category vs dip morphology** — 코드 2(loop skip) 와 코드 3(1-byte fault) 의 dip 모양이 *통계적으로 구분 가능* 한가?
- **ESP32 / STM32 RDP 등 실제 타겟 확장** — 본 노트북의 점퍼/디커플 가이드를 base 로 Raelize·SEC Consult 의 공격을 재현
- **EM-FI 와의 비교** — 동일 fault 가 EMFI(ChipSHOUTER) 에서는 어떻게 보이는가? (Husky 트리거 출력을 ChipSHOUTER 로 cascade)

---
*결합 위협 모델 — Husky VDD 와이어태핑 + Lite 전압 글리치 노트북 끝*
